In [1]:
from datasets import load_dataset
import pandas as pd
from tqdm.notebook import tqdm
import ahocorasick
import os
from datetime import datetime
import glob

CHECKPOINT_EVERY = 500_000
OUTPUT_DIR = "data_preselection"
FINAL_OUTPUT = "mappool_74M_scores_full.parquet"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✓ Konfiguracja załadowana")

✓ Konfiguracja załadowana


In [3]:
import json

with open("../stage1/keywords/positive_keywords.json", "r", encoding="utf-8") as f:
    positive_json = json.load(f)

with open("../stage1/keywords/negative_keywords.json", "r", encoding="utf-8") as f:
    negative_json = json.load(f)


print(f"Loaded {len(positive_json)} positive keywords.")
print(f"Loaded {len(negative_json)} negative keywords.")

print("Budowanie automatu Aho-Corasick...")

A = ahocorasick.Automaton()

for phrase, value in positive_json.items():
    A.add_word(phrase.lower(), ("pos", phrase, value))

for phrase, value in negative_json.items():
    A.add_word(phrase.lower(), ("neg", phrase, value))

A.make_automaton()

print(f"Automat gotowy: {len(positive_json)} pozytywnych, {len(negative_json)} negatywnych fraz")

Loaded 2725 positive keywords.
Loaded 2415 negative keywords.
Budowanie automatu Aho-Corasick...
Automat gotowy: 2725 pozytywnych, 2415 negatywnych fraz


In [4]:
def compute_score_full_words(text):
    if not isinstance(text, str):
        text = str(text)

    text_lower = text.lower()
    score = 0
    used_phrases = set()
    pos_keys = []
    neg_keys = []

    for end_index, (ptype, phrase, value) in A.iter(text_lower):
        start_index = end_index - len(phrase) + 1

        if (start_index == 0 or not text_lower[start_index-1].isalnum()) and \
           (end_index == len(text_lower)-1 or not text_lower[end_index+1].isalnum()):

            if phrase not in used_phrases:
                score += value
                used_phrases.add(phrase)
                if ptype == "pos":
                    pos_keys.append(phrase)
                else:
                    neg_keys.append(phrase)

    return score, pos_keys, neg_keys

def get_last_checkpoint():
    """Sprawdza od którego miejsca wznowić"""
    checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint_*.parquet')))

    if not checkpoints:
        print("📌 Brak checkpointów - rozpoczynam od początku")
        return 0, []

    last_checkpoint = checkpoints[-1]
    filename = os.path.basename(last_checkpoint)
    start_idx = int(filename.split('_rows_')[1].split('.')[0])

    print(f"📌 Znaleziono {len(checkpoints)} checkpointów")
    print(f"📌 Ostatni: {filename}")
    print(f"📌 Wczytywanie dotychczasowych wyników...")

    dfs = [pd.read_parquet(cp) for cp in tqdm(checkpoints, desc="Wczytywanie")]
    existing_df = pd.concat(dfs, ignore_index=True)
    existing_results = existing_df.to_dict('records')

    print(f"✓ Wznowienie od rekordu: {start_idx:,}")
    print(f"✓ Dotychczas przetworzonych: {len(existing_results):,} rekordów")

    return start_idx, existing_results


def save_checkpoint(results, checkpoint_num, count):
    """Zapisuje checkpoint"""
    checkpoint_file = os.path.join(
        OUTPUT_DIR,
        f'checkpoint_{checkpoint_num:03d}_rows_{count}.parquet'
    )

    start_of_batch = count - CHECKPOINT_EVERY
    batch_results = [r for r in results if r['idx'] >= start_of_batch]

    df_checkpoint = pd.DataFrame(batch_results)
    df_checkpoint.to_parquet(checkpoint_file, compression='snappy')

    return checkpoint_file


def print_stats(results, count, checkpoint_file):
    """Wyświetla statystyki"""
    df_stats = pd.DataFrame(results)

    print(f"\n{'='*70}")
    print(f"CHECKPOINT - Przetworzono {count:,} rekordów")
    print(f"Zapisano: {os.path.basename(checkpoint_file)}")
    print(f"\nStatystyki:")
    print(f"  Średni score: {df_stats['score'].mean():.2f}")
    print(f"  Score > 5:  {(df_stats['score'] > 5).sum():,} ({(df_stats['score'] > 5).sum()/len(df_stats)*100:.1f}%)")
    print(f"  Score 2-5:  {((df_stats['score'] > 2) & (df_stats['score'] <= 5)).sum():,}")
    print(f"  Score -1-2: {((df_stats['score'] >= -1) & (df_stats['score'] <= 2)).sum():,}")
    print(f"  Score < -1: {(df_stats['score'] < -1).sum():,}")
    print(f"  Postęp: {count/74_000_000*100:.1f}%")
    print(f"  Rozmiar: {os.path.getsize(checkpoint_file) / 1024**2:.1f} MB")
    print(f"{'='*70}\n")

print("✓ Funkcje załadowane")

✓ Funkcje załadowane


In [ ]:
# ============================================================================
# CELL 4: GŁÓWNA PĘTLA - MOŻESZ URUCHAMIAĆ WIELOKROTNIE!
# ============================================================================
#
# Ta cella:
# - Automatycznie wznawia od ostatniego checkpointu
# - Możesz ją przerwać (Kernel → Interrupt) i uruchomić ponownie
# - Możesz zamknąć notebook i wrócić następnego dnia
#
# ============================================================================

print(f"{'='*70}")
print(f"START: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}\n")

start_idx, existing_results = get_last_checkpoint()

print("\nŁadowanie datasetu MapPool...")
startTimeAll = datetime.now()
dataset = load_dataset("sraimund/MapPool", split="train", streaming=True)
endTimeLoadDataset = datetime.now()

print(f"✓ Dataset załadowany w {endTimeLoadDataset - startTimeAll}\n")

results = existing_results.copy()
count = start_idx
checkpoint_count = start_idx // CHECKPOINT_EVERY

if start_idx > 0:
    print(f"⏭️  Pomijam pierwsze {start_idx:,} rekordów...")
    dataset = dataset.skip(start_idx)
    print("✓ Gotowe!\n")

print(f"Rozpoczynam skanowanie od rekordu {count:,}...\n")

try:
    progress_bar = tqdm(
        dataset,
        total=75_905_000 - start_idx,
        initial=0,
        desc="Skanowanie",
        unit=" rekordów"
    )

    for row in progress_bar:
        text = row.get('text', '')
        score, pos_keys, neg_keys = compute_score_full_words(text)

        results.append({
            'idx': count,
            'url': row.get('url', ''),
            'score': score,
            'pos_keys': '|'.join(pos_keys) if pos_keys else '',
            'neg_keys': '|'.join(neg_keys) if neg_keys else '',
        })

        count += 1

        if count % CHECKPOINT_EVERY == 0:
            checkpoint_count += 1
            checkpoint_file = save_checkpoint(results, checkpoint_count, count)
            print_stats(results, count, checkpoint_file)

    print(f"\n{'='*70}")
    print(f"✓ KONIEC! Przetworzono wszystkie 74M rekordów")
    print(f"{'='*70}\n")

except KeyboardInterrupt:
    print("\n\n⚠️  PRZERWANO (Kernel Interrupt)")
    print(f"📌 Ostatni przetworzony rekord: {count:,}")
    print(f"📌 Checkpointy w: {OUTPUT_DIR}/")
    print(f"\n💡 Aby kontynuować, po prostu uruchom tę cellę ponownie!")

except Exception as e:
    print(f"\n\n❌ BŁĄD: {e}")
    import traceback
    traceback.print_exc()

START: 2026-01-31 23:12:03

📌 Brak checkpointów - rozpoczynam od początku

Ładowanie datasetu MapPool...


Resolving data files:   0%|          | 0/23876 [00:00<?, ?it/s]

✓ Dataset załadowany w 0:00:20.856988

Rozpoczynam skanowanie od rekordu 0...



Skanowanie:   0%|          | 0/75905000 [00:00<?, ? rekordów/s]

In [ ]:
# ============================================================================
# CELL 5: Finalne zapisanie (uruchom gdy zakończysz wszystkie 74M)
# ============================================================================

print("Wczytywanie wszystkich checkpointów i tworzenie finalnego pliku...")

checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint_*.parquet')))

if not checkpoints:
    print("❌ Brak checkpointów!")
else:
    dfs = [pd.read_parquet(cp) for cp in tqdm(checkpoints, desc="Wczytywanie")]
    df_final = pd.concat(dfs, ignore_index=True)

    df_final.to_parquet(FINAL_OUTPUT, compression='snappy')

    file_size_mb = os.path.getsize(FINAL_OUTPUT) / 1024**2

    print(f"\n{'='*70}")
    print(f"FINALNE WYNIKI")
    print(f"{'='*70}")
    print(f"Plik: {FINAL_OUTPUT}")
    print(f"Rozmiar: {file_size_mb:.1f} MB ({file_size_mb/1024:.2f} GB)")
    print(f"Rekordów: {len(df_final):,}")
    print(f"\nRozkład score'ów:")
    print(df_final['score'].describe())
    print(f"\nKategorie:")
    print(f"  Very positive (>5):  {(df_final['score'] > 5).sum():,} ({(df_final['score'] > 5).sum()/len(df_final)*100:.1f}%)")
    print(f"  Positive (2-5):      {((df_final['score'] > 2) & (df_final['score'] <= 5)).sum():,}")
    print(f"  Neutral (-1 to 2):   {((df_final['score'] >= -1) & (df_final['score'] <= 2)).sum():,}")
    print(f"  Negative (-3 to -1): {((df_final['score'] >= -3) & (df_final['score'] < -1)).sum():,}")
    print(f"  Very negative (<-3): {(df_final['score'] < -3).sum():,}")
    print(f"{'='*70}\n")

    print("✓ GOTOWE!")

In [ ]:
# ============================================================================
# CELL 6: Sprawdź obecny postęp (możesz uruchamiać w dowolnym momencie)
# ============================================================================

checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint_*.parquet')))

if not checkpoints:
    print("Brak checkpointów - skanowanie jeszcze nie rozpoczęte")
else:
    last_cp = checkpoints[-1]
    filename = os.path.basename(last_cp)
    current_row = int(filename.split('_rows_')[1].split('.')[0])

    progress_pct = current_row / 75_900_000 * 100

    print(f"\n{'='*70}")
    print(f"OBECNY POSTĘP")
    print(f"{'='*70}")
    print(f"Checkpointów: {len(checkpoints)}")
    print(f"Ostatni: {filename}")
    print(f"Przetworzonych: {current_row:,} / 74,000,000")
    print(f"Postęp: {progress_pct:.1f}%")

    # Szacowany czas
    remaining_rows = 75_900_000 - current_row
    estimated_seconds = (remaining_rows / 100_000) * 21
    estimated_hours = estimated_seconds / 3600

    print(f"Szacowany czas: ~{estimated_hours:.1f}h")

    # Quick stats
    df_last = pd.read_parquet(last_cp)
    print(f"\nOstatni batch:")
    print(f"  Średni score: {df_last['score'].mean():.2f}")
    print(f"  Pozytywne (>2): {(df_last['score'] > 2).sum():,}")

    # Timestamp
    from datetime import datetime
    last_modified = os.path.getmtime(last_cp)
    last_time = datetime.fromtimestamp(last_modified)
    print(f"\nOstatnia aktualizacja: {last_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}\n")